# Step 2 ? Calibrated AQI early-warning alerts

This notebook improves the interpretation of the nationwide AQI >200 and AQI >300 warning probabilities. It uses three distinct future periods after model training:

- **Calibration:** May?July 2024 ? fits Platt/sigmoid probability calibration.
- **Policy tuning:** August?October 2024 ? selects operating alert thresholds.
- **Final test:** November 2024?April 2025 ? evaluates calibration and false alarms once.

This prevents the final test outcomes from influencing probability calibration or threshold selection. NASA POWER remains historical reanalysis, so these are retrospective warning experiments rather than live weather-forecast results.


## 1. Imports and configuration

In [ ]:
from pathlib import Path
from time import perf_counter
import json
import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score, brier_score_loss, confusion_matrix, precision_score,
    recall_score, roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)

PROJECT_ROOT = Path(r"C:\Users\cdipi\Downloads\aqi_for india")
FULL_PATH = PROJECT_ROOT / "data" / "processed" / "aqi_prediction_full.csv"
REPORT_DIR = PROJECT_ROOT / "reports" / "calibration"
MODEL_DIR = PROJECT_ROOT / "models" / "calibration"
REPORT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)
assert FULL_PATH.exists(), FULL_PATH

HORIZONS = [1, 2, 3]
CUT_OFFS = [200, 300]
TRAIN_END = pd.Timestamp("2024-04-30")
CALIBRATION_END = pd.Timestamp("2024-07-31")
TUNING_END = pd.Timestamp("2024-10-31")
TEST_END = pd.Timestamp("2025-04-27")
TARGET_RECALL = 0.80
MAX_FALSE_POSITIVE_RATE = 0.05
COST_MULTIPLIER = 2.0
N_ESTIMATORS = 300
RANDOM_STATE = 42

print(f"Using: {FULL_PATH}")


## 2. Create leakage-safe nationwide forecasting features

AQI at the issue time and historical lags are allowed. `air_quality_status` and `prominent_pollutants` are excluded because they are same-day AQI-derived reporting fields. Future AQI values become labels only.


In [ ]:
WEATHER = [
    "temperature_2m_mean", "temperature_2m_max", "temperature_2m_min",
    "relative_humidity_2m_mean", "precipitation_sum", "wind_speed_2m_mean",
    "wind_speed_2m_max", "surface_pressure_mean", "shortwave_radiation_sum",
]

def make_event_table(path: Path) -> tuple[pd.DataFrame, list[str], list[str]]:
    data = pd.read_csv(path, parse_dates=["date"]).copy()
    weather = [column for column in WEATHER if column in data.columns]
    data["location_key"] = data["state"].astype(str).str.strip() + "||" + data["area"].astype(str).str.strip()
    data = data.sort_values(["location_key", "date"]).drop_duplicates(["location_key", "date"]).reset_index(drop=True)
    data = data.rename(columns={"aqi_value": "aqi_at_issue"})
    group = data.groupby("location_key", sort=False)

    for lag in (1, 3, 7, 14):
        feature = f"aqi_lag_{lag}"
        data[feature] = group["aqi_at_issue"].shift(lag)
        feature_date = group["date"].shift(lag)
        data.loc[(data.date - feature_date).dt.days.ne(lag), feature] = np.nan
    previous_aqi = group["aqi_at_issue"].shift(1)
    for window in (3, 7, 14):
        data[f"aqi_mean_{window}d"] = previous_aqi.groupby(data.location_key).transform(lambda s: s.rolling(window, min_periods=window).mean())
        data[f"aqi_max_{window}d"] = previous_aqi.groupby(data.location_key).transform(lambda s: s.rolling(window, min_periods=window).max())
        first_date = group.date.shift(window)
        invalid = (data.date - first_date).dt.days.ne(window)
        data.loc[invalid, [f"aqi_mean_{window}d", f"aqi_max_{window}d"]] = np.nan

    for column in weather:
        for lag in (1, 3, 7):
            feature = f"{column}_lag_{lag}"
            data[feature] = group[column].shift(lag)
            feature_date = group.date.shift(lag)
            data.loc[(data.date - feature_date).dt.days.ne(lag), feature] = np.nan

    for horizon in HORIZONS:
        future_date = data.date + pd.Timedelta(days=horizon)
        data[f"forecast_month_h{horizon}"] = future_date.dt.month
        data[f"forecast_doy_sin_h{horizon}"] = np.sin(2 * np.pi * future_date.dt.dayofyear / 365.25)
        data[f"forecast_doy_cos_h{horizon}"] = np.cos(2 * np.pi * future_date.dt.dayofyear / 365.25)
        target = group["aqi_at_issue"].shift(-horizon)
        target_date = group["date"].shift(-horizon)
        data[f"target_aqi_h{horizon}"] = target.where((target_date - data.date).dt.days.eq(horizon))
        for cutoff in CUT_OFFS:
            label = f"event_gt_{cutoff}_h{horizon}"
            data[label] = (data[f"target_aqi_h{horizon}"] > cutoff).astype(float)
            data.loc[data[f"target_aqi_h{horizon}"].isna(), label] = np.nan

    numeric = ["aqi_at_issue", "latitude", "longitude"]
    numeric += [c for c in data.columns if c.startswith(("aqi_lag_", "aqi_mean_", "aqi_max_"))]
    numeric += [c for c in data.columns if any(c.startswith(f"{w}_lag_") for w in weather)]
    numeric += [c for c in data.columns if c.startswith(("forecast_month_", "forecast_doy_"))]
    return data, numeric, ["state", "location_key"]

frame, NUMERIC, CATEGORICAL = make_event_table(FULL_PATH)
print(f"Rows: {len(frame):,}; named areas: {frame.area.nunique()}; state-area keys: {frame.location_key.nunique()}")
print(f"Features: {len(NUMERIC)} numeric + {len(CATEGORICAL)} categorical")


## 3. Probability calibration, threshold policies, and metrics

Platt scaling fits a logistic calibration model to probabilities produced in the calibration period. Three alert policies are then selected from the later tuning period:

1. **Safety-first:** maximum precision while recall is at least 0.80.
2. **Balanced F1:** threshold that maximizes F1.
3. **Low-false-alarm:** maximum recall subject to false-positive rate ?5%.

These policies make the recall/false-alarm trade-off explicit.


In [ ]:
def make_preprocessor() -> ColumnTransformer:
    return ColumnTransformer([
        ("numeric", Pipeline([(
            "imputer", SimpleImputer(strategy="median", add_indicator=True)
        )]), NUMERIC),
        ("categorical", Pipeline([(
            "imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore")),
        ]), CATEGORICAL),
    ])

def make_model(scale_pos_weight: float) -> Pipeline:
    return Pipeline([
        ("prep", make_preprocessor()),
        ("model", XGBClassifier(
            objective="binary:logistic", eval_metric="logloss", n_estimators=N_ESTIMATORS,
            max_depth=6, learning_rate=0.05, subsample=0.85, colsample_bytree=0.8,
            min_child_weight=3, reg_alpha=0.1, reg_lambda=1.0, scale_pos_weight=scale_pos_weight,
            tree_method="hist", n_jobs=-1, random_state=RANDOM_STATE,
        )),
    ])

def ece(y_true: np.ndarray, probability: np.ndarray, bins: int = 10) -> float:
    edges = np.linspace(0, 1, bins + 1)
    total = len(y_true)
    score = 0.0
    for start, end in zip(edges[:-1], edges[1:]):
        mask = (probability >= start) & ((probability < end) if end < 1 else (probability <= end))
        if mask.any():
            score += mask.mean() * abs(y_true[mask].mean() - probability[mask].mean())
    return float(score)

def threshold_metrics(y_true: np.ndarray, probability: np.ndarray, threshold: float) -> dict:
    issued = probability >= threshold
    tn, fp, fn, tp = confusion_matrix(y_true, issued, labels=[0, 1]).ravel()
    recall = recall_score(y_true, issued, zero_division=0)
    precision = precision_score(y_true, issued, zero_division=0)
    fpr = fp / (fp + tn) if fp + tn else np.nan
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return {
        "threshold": float(threshold), "Recall": recall, "Precision": precision, "F1": f1,
        "false_positive_rate": fpr, "alerts_issued": int(issued.sum()),
        "TP": int(tp), "FP": int(fp), "FN": int(fn), "TN": int(tn),
    }

def select_policy_thresholds(y_true: np.ndarray, probability: np.ndarray) -> dict[str, dict]:
    """Select policies on tuning data and retain the achieved tuning metrics.

    An FPR constraint is a tuning target, not a guarantee on unseen test dates.
    Keeping both values prevents the final report from overstating the policy.
    """
    candidates = np.round(np.arange(0.01, 1.00, 0.01), 2)
    metrics = pd.DataFrame([threshold_metrics(y_true, probability, t) for t in candidates])

    safety = metrics[metrics.Recall >= TARGET_RECALL]
    safety_choice = (safety if not safety.empty else metrics).sort_values(
        ["Precision", "threshold"], ascending=[False, False]
    ).iloc[0]
    f1_choice = metrics.sort_values(["F1", "threshold"], ascending=[False, False]).iloc[0]

    low_fpr = metrics[metrics.false_positive_rate <= MAX_FALSE_POSITIVE_RATE]
    if low_fpr.empty:
        raise RuntimeError(
            f"No tuning threshold meets FPR <= {MAX_FALSE_POSITIVE_RATE:.0%}; "
            "do not label any policy as FPR-constrained."
        )
    low_fpr_choice = low_fpr.sort_values(
        ["Recall", "Precision", "threshold"], ascending=[False, False, False]
    ).iloc[0]

    choices = {
        "Safety-first (recall target on tuning)": safety_choice,
        "Balanced F1 (tuning)": f1_choice,
        f"FPR-constrained tuning (target <= {MAX_FALSE_POSITIVE_RATE:.0%})": low_fpr_choice,
    }
    return {
        policy: {
            "threshold": float(choice.threshold),
            "tuning_Recall": float(choice.Recall),
            "tuning_Precision": float(choice.Precision),
            "tuning_F1": float(choice.F1),
            "tuning_false_positive_rate": float(choice.false_positive_rate),
            "tuning_fpr_target_met": bool(choice.false_positive_rate <= MAX_FALSE_POSITIVE_RATE),
        }
        for policy, choice in choices.items()
    }

def reliability_table(y_true: np.ndarray, probability: np.ndarray, bins: int = 10) -> pd.DataFrame:
    values = pd.DataFrame({"observed": y_true, "probability": probability})
    values["bin"] = pd.cut(values.probability, bins=np.linspace(0, 1, bins + 1), include_lowest=True)
    return values.groupby("bin", observed=True).agg(
        mean_probability=("probability", "mean"), event_rate=("observed", "mean"), count=("observed", "size")
    ).reset_index()


## 4. Train, calibrate, choose policies, and evaluate final-test alerts

In [ ]:
rows, reliability_rows, final_predictions = [], [], []
for horizon in HORIZONS:
    usable = frame[frame[f"target_aqi_h{horizon}"].notna()].copy()
    train = usable[usable.date <= TRAIN_END]
    calibration = usable[(usable.date > TRAIN_END) & (usable.date <= CALIBRATION_END)]
    tuning = usable[(usable.date > CALIBRATION_END) & (usable.date <= TUNING_END)]
    test = usable[(usable.date > TUNING_END) & (usable.date <= TEST_END)]

    for cutoff in CUT_OFFS:
        label = f"event_gt_{cutoff}_h{horizon}"
        y_train = train[label].astype(int).to_numpy()
        y_calibration = calibration[label].astype(int).to_numpy()
        y_tuning = tuning[label].astype(int).to_numpy()
        y_test = test[label].astype(int).to_numpy()
        if min(y_train.sum(), y_calibration.sum(), y_tuning.sum(), y_test.sum()) == 0:
            continue

        weight = ((len(y_train) - y_train.sum()) / y_train.sum()) * COST_MULTIPLIER
        model = make_model(weight)
        started = perf_counter()
        model.fit(train[NUMERIC + CATEGORICAL], y_train)
        train_seconds = perf_counter() - started

        calibration_raw = model.predict_proba(calibration[NUMERIC + CATEGORICAL])[:, 1]
        # Platt scaling is fitted only on dates that are later than model training.
        platt = LogisticRegression(C=1.0, solver="lbfgs", max_iter=1000, class_weight=None)
        platt.fit(calibration_raw.reshape(-1, 1), y_calibration)

        tuning_raw = model.predict_proba(tuning[NUMERIC + CATEGORICAL])[:, 1]
        tuning_calibrated = platt.predict_proba(tuning_raw.reshape(-1, 1))[:, 1]
        test_raw = model.predict_proba(test[NUMERIC + CATEGORICAL])[:, 1]
        test_calibrated = platt.predict_proba(test_raw.reshape(-1, 1))[:, 1]
        policies = select_policy_thresholds(y_tuning, tuning_calibrated)
        task = f"AQI_gt_{cutoff}_t_plus_{horizon}"

        joblib.dump({
            "base_model": model, "platt_calibrator": platt, "features": NUMERIC + CATEGORICAL,
            "policies": policies, "cutoff": cutoff, "horizon_days": horizon,
        }, MODEL_DIR / f"calibrated_{task}.joblib")

        common = {
            "task": task, "horizon_days": horizon, "AQI_cutoff": cutoff,
            "train_seconds": train_seconds, "scale_pos_weight": weight,
            "test_ROC_AUC": roc_auc_score(y_test, test_calibrated),
            "test_PR_AUC": average_precision_score(y_test, test_calibrated),
            "test_brier_raw": brier_score_loss(y_test, test_raw),
            "test_brier_calibrated": brier_score_loss(y_test, test_calibrated),
            "test_ECE_raw": ece(y_test, test_raw),
            "test_ECE_calibrated": ece(y_test, test_calibrated),
        }
        for policy, selection in policies.items():
            threshold = selection["threshold"]
            test_metrics = threshold_metrics(y_test, test_calibrated, threshold)
            rows.append({
                **common, "policy": policy, **selection, **test_metrics,
                "test_fpr_target_met": bool(test_metrics["false_positive_rate"] <= MAX_FALSE_POSITIVE_RATE),
            })
            predicted = test[["date", "state", "area", "location_key", f"target_aqi_h{horizon}"]].copy()
            predicted["forecast_date"] = predicted.date + pd.Timedelta(days=horizon)
            predicted["task"] = task
            predicted["policy"] = policy
            predicted["event_observed"] = y_test
            predicted["raw_probability"] = test_raw
            predicted["calibrated_probability"] = test_calibrated
            predicted["alert_threshold"] = threshold
            predicted["alert_issued"] = (test_calibrated >= threshold).astype(int)
            final_predictions.append(predicted)

        for probability_type, probability in [("raw", test_raw), ("calibrated", test_calibrated)]:
            table = reliability_table(y_test, probability)
            table["task"] = task
            table["probability_type"] = probability_type
            reliability_rows.append(table)

metrics_table = pd.DataFrame(rows).sort_values(["AQI_cutoff", "horizon_days", "policy"])
reliability_df = pd.concat(reliability_rows, ignore_index=True)
predictions_df = pd.concat(final_predictions, ignore_index=True)
metrics_table.to_csv(REPORT_DIR / "calibrated_alert_policy_metrics.csv", index=False)
reliability_df.to_csv(REPORT_DIR / "reliability_tables.csv", index=False)
predictions_df.to_csv(REPORT_DIR / "calibrated_alert_test_predictions.csv", index=False)
display(metrics_table.round(3))


## 5. Calibration and policy plots

A calibrated model should make probabilities correspond more closely to observed event frequencies. Compare raw and calibrated reliability curves, then use the policy table to choose an alert strategy with acceptable missed-event and false-alert rates.


In [ ]:
for task in reliability_df.task.unique():
    subset = reliability_df[reliability_df.task.eq(task)]
    plt.figure(figsize=(5, 5))
    for kind, style in [("raw", "--"), ("calibrated", "-")]:
        curve = subset[subset.probability_type.eq(kind)]
        plt.plot(curve.mean_probability, curve.event_rate, marker="o", linestyle=style, label=kind)
    plt.plot([0, 1], [0, 1], "k:", label="perfect calibration")
    plt.xlim(0, 1); plt.ylim(0, 1)
    plt.xlabel("Mean predicted probability")
    plt.ylabel("Observed event rate")
    plt.title(f"Reliability diagram ? {task}")
    plt.legend()
    plt.tight_layout()
    plt.savefig(REPORT_DIR / f"reliability_{task}.png", dpi=160)
    plt.show()

plt.figure(figsize=(12, 5))
sns.barplot(data=metrics_table, x="task", y="false_positive_rate", hue="policy")
plt.axhline(MAX_FALSE_POSITIVE_RATE, color="black", linestyle="--", linewidth=1, label="Tuning FPR target")
plt.ylim(0, 1); plt.xticks(rotation=20, ha="right")
plt.title("Final-test false-positive rate by alert policy")
plt.legend()
plt.tight_layout()
plt.savefig(REPORT_DIR / "policy_false_positive_rates.png", dpi=160)
plt.show()

plt.figure(figsize=(12, 5))
sns.barplot(data=metrics_table, x="task", y="Recall", hue="policy")
plt.ylim(0, 1); plt.xticks(rotation=20, ha="right")
plt.title("Final-test recall by alert policy")
plt.tight_layout()
plt.savefig(REPORT_DIR / "policy_recall.png", dpi=160)
plt.show()


## 6. Select an operating alert policy

Policies are selected only from the tuning period. The FPR-constrained policy targets the specified false-positive rate **on tuning data**; its final-test FPR is reported separately and may differ because later conditions can shift. Select a deployment policy prospectively based on the organisation's tolerance for missed events versus false alarms. The safety-first policy is appropriate for internal screening, while public alerts require an independently acceptable final-test false-alert rate.


In [ ]:
summary_columns = [
    "task", "policy", "threshold", "tuning_Recall", "tuning_Precision",
    "tuning_false_positive_rate", "tuning_fpr_target_met", "Recall", "Precision",
    "false_positive_rate", "test_fpr_target_met", "alerts_issued", "TP", "FP", "FN",
    "test_brier_raw", "test_brier_calibrated", "test_ECE_raw", "test_ECE_calibrated",
]
display(metrics_table[summary_columns].round(3))

manifest = {
    "input": str(FULL_PATH.relative_to(PROJECT_ROOT)),
    "train_end": str(TRAIN_END.date()), "calibration_end": str(CALIBRATION_END.date()),
    "policy_tuning_end": str(TUNING_END.date()), "test_end": str(TEST_END.date()),
    "calibration_method": "Platt scaling (logistic regression on validation probabilities)",
    "target_recall": TARGET_RECALL, "max_false_positive_rate": MAX_FALSE_POSITIVE_RATE,
}
with (REPORT_DIR / "calibration_manifest.json").open("w", encoding="utf-8") as handle:
    json.dump(manifest, handle, indent=2)
print("Saved reports:")
for output in sorted(REPORT_DIR.iterdir()):
    print(" -", output.relative_to(PROJECT_ROOT))
